In [ ]:
import pandas as pd
import numpy as np
import re
import glob

In [ ]:
df = pd.read_csv("../raw_data/amazon_india_2022.csv")

In [3]:
df.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2022_00000001,2022-01-22,CUST_2022_00015913,PROD_000527,Samsung Galaxy A50 64GB White,Electronics,Smartphones,Samsung,26992.07,41.06,...,True,Republic Day Sale,4.0,Delivered,1,2022,1,0.20,0,4.2
1,TXN_2022_00000002,2022-01-11,CUST_2022_00021843,PROD_000979,Samsung Galaxy S22+ 128GB White,Electronics,Smartphones,Samsung,80105.77,0.00,...,False,NaN,NaN,Delivered,1,2022,1,0.24,False,3.3
2,TXN_2022_00000003,17-01-2022,CUST_2019_00043917,PROD_001909,Xiaomi Fitness Band Deluxe,Electronics,Smart Watch,Xiaomi,27516.92,0.00,...,False,NaN,4.0,Delivered,1,2022,1,0.05,True,4.2
3,TXN_2022_00000004,2022-01-27,CUST_2022_00011696,PROD_000284,Xiaomi Mi A1 32GB White,Electronics,Smartphones,Xiaomi,38018.11,0.00,...,FALSE,NaN,4.0,Delivered,1,2022,1,0.24,True,4.6
4,TXN_2022_00000005,2022-01-16,CUST_2022_00030756,PROD_000556,OnePlus OnePlus 7T 64GB Gold,Electronics,Smartphones,OnePlus,87743.24,0.00,...,False,NaN,NaN,Delivered,1,2022,1,0.18,True,3.4


In [4]:
df["delivery_charges"].isna().sum(), len(df)

(np.int64(10616), 132660)

In [5]:
df["delivery_charges"].describe()

count    122044.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: delivery_charges, dtype: float64

In [6]:
df["delivery_charges"].isna().sum()

np.int64(10616)

In [7]:
df.drop(columns=["delivery_charges"], inplace=True)

In [8]:
df.shape

(132660, 33)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132660 entries, 0 to 132659
Data columns (total 33 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   transaction_id          132660 non-null  object 
 1   order_date              132660 non-null  object 
 2   customer_id             132660 non-null  object 
 3   product_id              132660 non-null  object 
 4   product_name            132660 non-null  object 
 5   category                132660 non-null  object 
 6   subcategory             132660 non-null  object 
 7   brand                   132660 non-null  object 
 8   original_price_inr      132660 non-null  object 
 9   discount_percent        132660 non-null  float64
 10  discounted_price_inr    132660 non-null  float64
 11  quantity                132660 non-null  int64  
 12  subtotal_inr            132660 non-null  float64
 13  final_amount_inr        132660 non-null  float64
 14  customer_city       

In [10]:
df.columns

Index(['transaction_id', 'order_date', 'customer_id', 'product_id',
       'product_name', 'category', 'subcategory', 'brand',
       'original_price_inr', 'discount_percent', 'discounted_price_inr',
       'quantity', 'subtotal_inr', 'final_amount_inr', 'customer_city',
       'customer_state', 'customer_tier', 'customer_spending_tier',
       'customer_age_group', 'payment_method', 'delivery_days',
       'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name',
       'customer_rating', 'return_status', 'order_month', 'order_year',
       'order_quarter', 'product_weight_kg', 'is_prime_eligible',
       'product_rating'],
      dtype='object')

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [11]:
df["order_date"].head(20)

0     2022-01-22
1     2022-01-11
2     17-01-2022
3     2022-01-27
4     2022-01-16
5     2022-01-02
6     2022-01-16
7     2022-01-15
8     2022-01-01
9     2022-01-25
10    2022-01-26
11    2022-01-31
12    2022-01-20
13    2022-01-06
14    2022-01-17
15    2022-01-03
16    2022-01-22
17    2022-01-12
18    2022-01-22
19    2022-01-12
Name: order_date, dtype: object

In [12]:
df["order_date"] = (
    df["order_date"]
    .str.replace(" ", "", regex=False)
    .str.replace("/", "-", regex=False)
)

parts = df["order_date"].str.split("-", expand=True)

year_last = parts[2].str.len() == 4

df.loc[year_last, "order_date"] = (
    parts[2] + "-" + parts[0] + "-" + parts[1]
)

parts = df["order_date"].str.split("-", expand=True)

mask = parts[1].astype(int) > 12

df.loc[mask, "order_date"] = (
    parts[0] + "-" + parts[2] + "-" + parts[1]
)

df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

In [13]:
df["order_date"].min(), df["order_date"].max()

(Timestamp('2022-01-01 00:00:00'), Timestamp('2022-12-31 00:00:00'))

In [14]:
df["order_date"].isna().sum()

np.int64(0)

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees. 


In [15]:
df["original_price_inr"] = df["original_price_inr"].astype(str)

df["original_price_inr"] = df["original_price_inr"].str.replace("₹", "", regex=False)

df["original_price_inr"] = df["original_price_inr"].str.replace(",", "", regex=False)

df["original_price_inr"] = pd.to_numeric(df["original_price_inr"], errors="coerce")

In [16]:
df["original_price_inr"].unique()[:20]

array([ 26992.07,  80105.77,  27516.92,  38018.11,  87743.24,  23400.44,
        99959.4 ,  25683.54,  14668.02, 147614.65,  20683.47, 136516.62,
        43979.69,  39705.95,  75063.07,  49987.09,  36182.34,  28244.07,
             nan, 116292.79])

In [17]:
mask = df["original_price_inr"].isna()

df.loc[mask, "original_price_inr"] = np.where(
    df.loc[mask, "discount_percent"] == 0,
    
    # Case 1: no discount
    df.loc[mask, "discounted_price_inr"],
    
    # Case 2: discount present
    df.loc[mask, "discounted_price_inr"] / (1 - df.loc[mask, "discount_percent"] / 100)
)

In [18]:
df["original_price_inr"].dtypes

dtype('float64')

In [19]:
df["original_price_inr"].isna().sum()

np.int64(0)

Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.


In [20]:
df["customer_rating"] = df["customer_rating"].astype(str)

df["customer_rating"] = df["customer_rating"].str.replace(" stars", "", regex=False)

df["customer_rating"] = df["customer_rating"].str.split("/").str[0]

df["customer_rating"] = pd.to_numeric(df["customer_rating"], errors="coerce")

In [21]:
df["customer_rating"].describe()

count    92454.000000
mean         4.303778
std          0.573599
min          3.000000
25%          4.000000
50%          4.500000
75%          5.000000
max          5.000000
Name: customer_rating, dtype: float64

In [22]:
df["customer_rating"].value_counts().head(10)

customer_rating
4.5    30288
4.0    23623
5.0    23354
3.5     9553
3.0     5636
Name: count, dtype: int64

In [23]:
df["customer_rating"].isna().sum()

np.int64(40206)

In [24]:
df["customer_rating"].value_counts(dropna=False)

customer_rating
NaN    40206
4.5    30288
4.0    23623
5.0    23354
3.5     9553
3.0     5636
Name: count, dtype: int64

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.


In [25]:
df["customer_city"] = df["customer_city"].str.strip().str.lower()

In [26]:
df["customer_city"].unique()

array(['vadodara', 'chennai', 'delhi', 'gorakhpur', 'ludhiana', 'pune',
       'lucknow', 'kanpur', 'indore', 'ahmedabad', 'kolkata', 'mumbai',
       'bangalore', 'coimbatore', 'nagpur', 'moradabad', 'jaipur',
       'visakhapatnam', 'saharanpur', 'patna', 'chandigarh', 'allahabad',
       'bhubaneswar', 'kochi', 'hyderabad', 'aligarh', 'surat',
       'bareilly', 'new delhi', 'varanasi', 'meerut', 'chenai',
       'calcutta', 'bombay', 'delhi ncr', 'mumba', 'madras', 'bengalore',
       'banglore', 'bengaluru'], dtype=object)

In [27]:
city_map = {
    "new delhi": "delhi",
    "delhi ncr": "delhi",

    "chenai": "chennai",
    "madras": "chennai",

    "calcutta": "kolkata",

    "bombay": "mumbai",
    "mumba": "mumbai",

    "bengalore": "bangalore",
    "banglore": "bangalore",
    "bengaluru": "bangalore"
}

In [28]:
df["customer_city"] = df["customer_city"].replace(city_map)

In [29]:
df["customer_city"] = df["customer_city"].str.title()

In [30]:
df["customer_city"].value_counts().head(20)

customer_city
Mumbai           15016
Delhi            13011
Bangalore        11111
Chennai           9159
Pune              8853
Kolkata           7196
Ahmedabad         6428
Surat             5216
Jaipur            5179
Nagpur            4817
Hyderabad         4804
Lucknow           4352
Kanpur            4302
Indore            4290
Coimbatore        3334
Kochi             3111
Bhubaneswar       2784
Vadodara          2702
Visakhapatnam     2693
Patna             2664
Name: count, dtype: int64

In [31]:
df["customer_city"].unique()

array(['Vadodara', 'Chennai', 'Delhi', 'Gorakhpur', 'Ludhiana', 'Pune',
       'Lucknow', 'Kanpur', 'Indore', 'Ahmedabad', 'Kolkata', 'Mumbai',
       'Bangalore', 'Coimbatore', 'Nagpur', 'Moradabad', 'Jaipur',
       'Visakhapatnam', 'Saharanpur', 'Patna', 'Chandigarh', 'Allahabad',
       'Bhubaneswar', 'Kochi', 'Hyderabad', 'Aligarh', 'Surat',
       'Bareilly', 'Varanasi', 'Meerut'], dtype=object)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [32]:
bool_candidates = []

bool_values = {"true","false","yes","no","y","n","1","0"}

for col in df.columns:
    vals = set(df[col].astype(str).str.lower().dropna().unique())
    
    if vals & bool_values:
        bool_candidates.append(col)

bool_candidates

['quantity',
 'delivery_days',
 'is_prime_member',
 'is_festival_sale',
 'order_month',
 'order_quarter',
 'is_prime_eligible']

In [33]:
boolean_cols = ["is_prime_member", "is_prime_eligible", "is_festival_sale"]

bool_map = {
    "true": True,
    "false": False,
    "yes": True,
    "no": False,
    "y": True,
    "n": False,
    "1": True,
    "0": False
}

for col in boolean_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(bool_map)
    )

In [34]:
df[boolean_cols].value_counts(dropna=False)

is_prime_member  is_prime_eligible  is_festival_sale
True             True               False               38510
False            True               False               37334
True             True               True                17172
False            True               True                16469
True             False              False                8305
False            False              False                7719
True             False              True                 3742
False            False              True                 3409
Name: count, dtype: int64

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.


In [35]:
df["category"].value_counts().head(20)

category
Electronics                  132580
Electronics & Accessories        22
ELECTRONICS                      22
Electronicss                     19
Electronic                       17
Name: count, dtype: int64

In [36]:
category_map = {
    "ELECTRONICS": "Electronics",
    "Electronics & Accessories": "Electronics",
    "Electronic": "Electronics",
    "Electronicss": "Electronics"
}

In [37]:
df["category"] = df["category"].replace(category_map)
df["category"] = df["category"].str.title()

In [38]:
df["category"].value_counts()

category
Electronics    132660
Name: count, dtype: int64

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [39]:
df["delivery_days"].unique()

array(['1', '2', '4', '3', '5', '-1', '6', '1-2 days', '7', 'Express',
       'Same Day', '15', '0'], dtype=object)

In [40]:
df["delivery_days"] = df["delivery_days"].astype(str).str.strip().str.lower()

In [41]:
df["delivery_days"] = df["delivery_days"].replace({
    "same day": "0",
    "express": "1"
})

In [42]:
df["delivery_days"] = df["delivery_days"].str.extract(r"(-?\d+)")

In [43]:
df["delivery_days"] = pd.to_numeric(df["delivery_days"], errors="coerce")

In [44]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = None

In [45]:
df["delivery_days"].unique()

array([ 1.,  2.,  4.,  3.,  5., nan,  6.,  7.,  0., 15.])

In [46]:
df["delivery_days"].isnull().sum()

np.int64(796)

In [47]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = np.nan

df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median())

In [48]:
df["delivery_days"].describe()

count    132660.000000
mean          2.985972
std           1.746859
min           0.000000
25%           1.000000
50%           3.000000
75%           4.000000
max          15.000000
Name: delivery_days, dtype: float64

In [49]:
df["delivery_days"].isna().sum()

np.int64(0)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [50]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [51]:
duplicates.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
452,TXN_2022_00000453,2022-01-02,CUST_2022_00023792,PROD_000011,Apple iPhone 6 Plus 64GB Black,Electronics,Smartphones,Apple,134850.36,0.00,...,False,NaN,4.0,Delivered,1,2022,1,0.23,True,3.8
668,TXN_2022_00000669,2022-01-21,CUST_2022_00009112,PROD_000486,Apple iPhone 11 128GB Blue,Electronics,Smartphones,Apple,125751.93,37.70,...,True,Republic Day Sale,4.5,Delivered,1,2022,1,0.19,True,4.3
744,TXN_2022_00000745,2022-01-15,CUST_2021_00016298,PROD_000874,Xiaomi Redmi Note 10 128GB Black,Electronics,Smartphones,Xiaomi,21463.27,0.00,...,False,NaN,4.0,Delivered,1,2022,1,0.15,True,3.5
1515,TXN_2022_00001516,2022-01-18,CUST_2018_00024264,PROD_001936,Garmin Band,Electronics,Smart Watch,Garmin,44454.98,0.00,...,False,NaN,4.5,Delivered,1,2022,1,0.05,True,4.2
1926,TXN_2022_00001927,2022-01-11,CUST_2022_00014977,PROD_000790,Oppo Reno 4 Pro 128GB White,Electronics,Smartphones,Oppo,30992.79,5.07,...,False,NaN,3.5,Delivered,1,2022,1,0.15,True,3.9


In [52]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [53]:
duplicates.shape

(1306, 33)

In [54]:
df.duplicated().sum()

np.int64(0)

In [55]:
duplicates.sort_values(
    ["customer_id","product_id","order_date"]
).head(10)

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
129743,TXN_2022_00129744,2022-12-18,CUST_2015_00000260,PROD_000407,Xiaomi Redmi 5A 256GB Black,Electronics,Smartphones,Xiaomi,39999.37,14.26,...,False,NaN,4.0,Delivered,12,2022,4,0.23,True,4.6
132315,TXN_2022_00129744_DUP,2022-12-18,CUST_2015_00000260,PROD_000407,Xiaomi Redmi 5A 256GB Black,Electronics,Smartphones,Xiaomi,39999.37,14.26,...,False,NaN,4.0,Delivered,12,2022,4,0.23,True,4.6
79931,TXN_2022_00079932,2022-08-30,CUST_2015_00000296,PROD_001743,Realme Galaxy Tab 8GB RAM Silver,Electronics,Tablets,Realme,70474.69,0.00,...,False,NaN,5.0,Delivered,8,2022,3,0.67,True,4.0
132335,TXN_2022_00079932_DUP,2022-08-30,CUST_2015_00000296,PROD_001743,Realme Galaxy Tab 8GB RAM Silver,Electronics,Tablets,Realme,70474.69,0.00,...,False,NaN,5.0,Delivered,8,2022,3,0.67,True,4.0
111610,TXN_2022_00111611,2022-11-07,CUST_2015_00000380,PROD_001957,Noise Sports Watch Premium,Electronics,Smart Watch,Noise,47284.22,19.35,...,False,NaN,4.5,Delivered,11,2022,4,0.03,False,4.2
132280,TXN_2022_00111611_DUP,2022-11-07,CUST_2015_00000380,PROD_001957,Noise Sports Watch Premium,Electronics,Smart Watch,Noise,47284.22,19.35,...,False,NaN,4.5,Delivered,11,2022,4,0.03,False,4.2
24418,TXN_2022_00024419,2022-03-08,CUST_2015_00002644,PROD_000389,OnePlus OnePlus 6T 64GB Black,Electronics,Smartphones,OnePlus,59533.02,22.91,...,True,Holi Festival,4.5,Delivered,3,2022,1,0.17,True,4.2
132049,TXN_2022_00024419_DUP,2022-03-08,CUST_2015_00002644,PROD_000389,OnePlus OnePlus 6T 64GB Black,Electronics,Smartphones,OnePlus,59533.02,22.91,...,True,Holi Festival,4.5,Delivered,3,2022,1,0.17,True,4.2
5023,TXN_2022_00005024,2022-01-24,CUST_2015_00002656,PROD_001776,Sony TWS,Electronics,Audio,Sony,9253.18,49.84,...,True,Republic Day Sale,4.5,Delivered,1,2022,1,0.11,True,4.3
132499,TXN_2022_00005024_DUP,2022-01-24,CUST_2015_00002656,PROD_001776,Sony TWS,Electronics,Audio,Sony,9253.18,49.84,...,True,Republic Day Sale,4.5,Delivered,1,2022,1,0.11,True,4.3


In [56]:
duplicates.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().sort_values(ascending=False).head(10)

customer_id         product_id   order_date  original_price_inr
CUST_2015_00000260  PROD_000407  2022-12-18  39999.37              2
CUST_2015_00000296  PROD_001743  2022-08-30  70474.69              2
CUST_2015_00000380  PROD_001957  2022-11-07  47284.22              2
CUST_2015_00002644  PROD_000389  2022-03-08  59533.02              2
CUST_2015_00002656  PROD_001776  2022-01-24  9253.18               2
CUST_2015_00003140  PROD_001710  2022-11-05  99959.40              2
CUST_2015_00003947  PROD_000162  2022-07-24  41250.59              2
CUST_2015_00006639  PROD_000383  2022-09-03  19883.17              2
CUST_2015_00011710  PROD_000172  2022-02-18  29495.03              2
CUST_2016_00002345  PROD_000230  2022-09-14  116292.79             2
dtype: int64

In [57]:
dup_groups = df.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().reset_index(name="count")

dup_groups = dup_groups[dup_groups["count"] > 1]

In [58]:
dup_rows = df.merge(
    dup_groups,
    on=["customer_id","product_id","order_date","original_price_inr"],
    how="inner"
)

In [59]:
dup_rows[["customer_id","product_id","quantity","count"]].head()

,customer_id,product_id,quantity,count
0,CUST_2022_00023792,PROD_000011,1,2
1,CUST_2022_00009112,PROD_000486,1,2
2,CUST_2021_00016298,PROD_000874,1,2
3,CUST_2018_00024264,PROD_001936,1,2
4,CUST_2022_00014977,PROD_000790,1,2


In [60]:
df_clean = df.drop_duplicates(
    subset=["customer_id","product_id","order_date","original_price_inr"],
    keep="first"
)

In [61]:
df_clean.duplicated(
    subset=["customer_id","product_id","order_date","original_price_inr"]
).sum()

np.int64(0)

In [62]:
df["transaction_id"].duplicated().sum()

np.int64(0)

In [63]:
df = df.drop_duplicates(subset="transaction_id", keep="first")

In [64]:
df[df["transaction_id"]=="TXN_2015_00000280"]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating


Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [65]:

# ── FIX 1: Negative prices ──────────────────────────
neg_mask = df["original_price_inr"] < 0
df.loc[neg_mask, "original_price_inr"] = df.loc[neg_mask, "original_price_inr"].abs()
print(f"Negative prices fixed: {neg_mask.sum()}")

# ── FIX 2: Outliers ──────────────────────────────────
subcategory_caps = {
    "Smart Watch":        75000,
    "Tablets":            155000,
    "Smartphones":        350000,    
    "Laptops":            260000,
    "TV & Entertainment": 300000,
    "Audio":              50000,
}

outlier_mask = df.apply(
    lambda row: row["original_price_inr"] > subcategory_caps.get(row["subcategory"], 999999),
    axis=1
)
df.loc[outlier_mask, "original_price_inr"] = (
    df.loc[outlier_mask, "original_price_inr"] / 100
).round(2)
print(f"Outliers fixed: {outlier_mask.sum()}")

# ── FIX 3: delivery_charges ──────────────────────────
df["delivery_charges"] = df["delivery_charges"].fillna(0)

# ── FIX 4: Recalculate ───────────────────────────────
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = (df["subtotal_inr"] + df["delivery_charges"]).round(2)

# ── VERIFY ───────────────────────────────────────────
print(df.groupby("subcategory", observed=True)["original_price_inr"]
      .describe()[["min","max","mean","50%"]].round(2))
print(f"\nNaN in final_amount_inr:   {df['final_amount_inr'].isna().sum()}")
print(f"Negative prices remaining: {(df['original_price_inr'] < 0).sum()}")


Negative prices fixed: 308
Outliers fixed: 562


KeyError: 'delivery_charges'

In [ ]:
# Check if any legitimate products were over-corrected in cleaned files
for sub, cap in subcategory_caps.items():
    over = df[(df["subcategory"] == sub) & 
                    (df["original_price_inr"] > cap)]
    if len(over) > 0:
        print(f"\n{sub} (cap ₹{cap:,}): {len(over)} rows over")
        print(over[["product_name", "original_price_inr"]].drop_duplicates().head(5))
    else:
        print(f"\n{sub}: ✅ All within cap")

Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 
'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [ ]:
# ── Standardize payment methods ────────────────────
payment_standardize = {
    # UPI variants
    "UPI": "UPI", "PhonePe": "UPI", "GooglePay": "UPI", "Google Pay": "UPI",
    "UPI/PhonePe": "UPI", "UPI/GooglePay": "UPI",

    # Credit Card variants
    "Credit Card": "Credit Card", "CREDIT_CARD": "Credit Card", "CC": "Credit Card",

    # Debit Card variants
    "Debit Card": "Debit Card", "DEBIT_CARD": "Debit Card", "DC": "Debit Card",

    # COD variants
    "Cash on Delivery": "COD", "COD": "COD", "C.O.D": "COD",

    # Others
    "Wallet": "Wallet",
    "Net Banking": "Net Banking",
    "BNPL": "BNPL"
}

df["payment_method"] = df["payment_method"].map(payment_standardize).fillna(df["payment_method"])

# ── Create categorical hierarchy ───────────────────
payment_category = {
    "UPI":          "Digital Payment",
    "Wallet":       "Digital Payment",
    "Net Banking":  "Digital Payment",
    "Credit Card":  "Card Payment",
    "Debit Card":   "Card Payment",
    "BNPL":         "Pay Later",
    "COD":          "Cash on Delivery"
}

df["payment_category"] = df["payment_method"].map(payment_category).astype("category")

# ── Verify ─────────────────────────────────────────
print(df["payment_method"].value_counts())
print(f"\nNaN in payment_category: {df['payment_category'].isna().sum()}")

payment_method
UPI            60848
COD            19972
Credit Card    19926
Debit Card     15947
Net Banking     8070
Wallet          5255
BNPL            2642
Name: count, dtype: int64

NaN in payment_category: 0


Handling Nan - in customer age group

In [ ]:
df["customer_age_group"] = df["customer_age_group"].fillna("Unknown")

Checking for object columns

In [ ]:
df.select_dtypes(include="object").columns

Index(['transaction_id', 'customer_id', 'product_id', 'product_name',
       'category', 'subcategory', 'brand', 'customer_city', 'customer_state',
       'customer_tier', 'customer_spending_tier', 'customer_age_group',
       'payment_method', 'delivery_type', 'festival_name', 'return_status'],
      dtype='object')

In [ ]:
# ── Optimize memory: convert to category dtype ───────
cat_columns = ["category", "subcategory", "customer_tier", 
               "customer_spending_tier", "customer_age_group",
               "payment_method", "delivery_type", 
               "festival_name", "return_status"]

df[cat_columns] = df[cat_columns].astype("category")

# Verify
print(df[cat_columns].dtypes)
print(f"\nMemory usage after optimization:")
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

category                  category
subcategory               category
customer_tier             category
customer_spending_tier    category
customer_age_group        category
payment_method            category
delivery_type             category
festival_name             category
return_status             category
dtype: object

Memory usage after optimization:
71.25021934509277 MB


In [ ]:
# NaN summary for all columns
nan_summary = df.isna().sum()
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False)

print(f"Total columns with NaN: {len(nan_summary)}")
print(f"Total rows in dataset: {df.shape[0]}")
print(f"\nNaN counts and percentages:")
print(pd.DataFrame({
    "NaN Count": nan_summary,
    "Percentage": (nan_summary / df.shape[0] * 100).round(2)
}))

In [66]:
df.to_csv("data_cleaning_2022.csv", index=False)
print("File saved successfully!")

File saved successfully!
